In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)

Device: cuda


In [2]:
# 셀 2 — Fashion-MNIST DataLoader

transform = transforms.ToTensor()

train_dataset = datasets.FashionMNIST(
    root="./data",
    train=True,
    transform=transform,
    download=True,
)

validation_dataset = datasets.FashionMNIST(
    root="./data",
    train=False,
    transform=transform,
    download=True,
)

batch_size = 256

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
)

validation_loader = DataLoader(
    dataset=validation_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
)

batch_images, batch_labels = next(
    iter(train_loader)
)

print("Image batch shape:", batch_images.shape)
print("Label batch shape:", batch_labels.shape)
print("Number of training batches:", len(train_loader))
print("Number of validation batches:", len(validation_loader))

Image batch shape: torch.Size([256, 1, 28, 28])
Label batch shape: torch.Size([256])
Number of training batches: 235
Number of validation batches: 40


In [3]:
# 셀 3 — 모델·손실함수·옵티마이저
class SoftmaxRegression(nn.Module):
    """PyTorch 고수준 API로 구현한 softmax 회귀 모델."""
    
    def __init__(
        self,
        num_outputs,
    ):
        super().__init__()
        
        self.net = nn.Sequential(
            # [B, 1, 28, 28] -> [B, 784]
            nn.Flatten(),
            
            # [B, 784] -> [B, num_outputs]
            nn.LazyLinear(num_outputs),
        )
        
    def forward(self, images):
        # Softmax를 적용하지 않은 logit을 반환한다.
        return self.net(images)
    
    
model = SoftmaxRegression(
    num_outputs=10,
).to(device)

# LazyLinear의 입력 feature 수를 확정
with torch.no_grad():
    demo_logits = model(
        batch_images[:1].to(device)
    )
    
loss_function = nn.CrossEntropyLoss()

learning_rete = 0.1

optimizer = torch.optim.SGD(
    params=model.parameters(),
    lr=learning_rete,
)

print(model)
print("\nDemo logits shape:", demo_logits.shape)

SoftmaxRegression(
  (net): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=784, out_features=10, bias=True)
  )
)

Demo logits shape: torch.Size([1, 10])


In [4]:
# validation function

@torch.no_grad()
def evaluate(
    model,
    data_loader,
    loss_function,
):
    model.eval()
    
    total_loss = 0.0
    total_correct = 0
    total_examples = 0
    
    for images, labels in data_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        logits = model(images)
        
        loss = loss_function(
            logits,
            labels,
        )
        
        predictions = logits.argmax(
            dim=-1,
        )
        
        current_batch_size = labels.shape[0]
        
        total_loss += (
            loss.item()
            * current_batch_size
        )
        
        total_correct += (
            predictions == labels
        ).sum().item()
        
        total_examples += current_batch_size

    mean_loss = total_loss / total_examples
    accuracy = total_correct / total_examples
    
    return mean_loss, accuracy

In [5]:
# 셀 5 — 모델 학습
num_epochs = 10
history = []

for epoch in range(num_epochs):
    model.train()

    total_train_loss = 0.0
    total_train_examples = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        # 이전 미니배치의 gradient를 제거한다.
        optimizer.zero_grad()

        # 모델의 logit을 계산한다.
        logits = model(images)

        # CrossEntropyLoss에 logit을 직접 전달한다.
        loss = loss_function(
            logits,
            labels,
        )

        # Gradient 계산과 parameter 갱신
        loss.backward()
        optimizer.step()

        current_batch_size = labels.shape[0]

        total_train_loss += (
            loss.item()
            * current_batch_size
        )

        total_train_examples += current_batch_size

    train_loss = (
        total_train_loss
        / total_train_examples
    )

    validation_loss, validation_accuracy = evaluate(
        model,
        validation_loader,
        loss_function,
    )

    history.append({
        "train_loss": train_loss,
        "validation_loss": validation_loss,
        "validation_accuracy": validation_accuracy,
    })

    print(
        f"Epoch {epoch + 1:02d} "
        f"| Train loss: {train_loss:.4f} "
        f"| Validation loss: {validation_loss:.4f} "
        f"| Validation accuracy: {validation_accuracy:.4f}"
    )

Epoch 01 | Train loss: 0.7879 | Validation loss: 0.6369 | Validation accuracy: 0.7880
Epoch 02 | Train loss: 0.5709 | Validation loss: 0.5755 | Validation accuracy: 0.8055
Epoch 03 | Train loss: 0.5259 | Validation loss: 0.5533 | Validation accuracy: 0.8083
Epoch 04 | Train loss: 0.5015 | Validation loss: 0.5252 | Validation accuracy: 0.8209
Epoch 05 | Train loss: 0.4862 | Validation loss: 0.5105 | Validation accuracy: 0.8238
Epoch 06 | Train loss: 0.4740 | Validation loss: 0.5116 | Validation accuracy: 0.8193
Epoch 07 | Train loss: 0.4645 | Validation loss: 0.5086 | Validation accuracy: 0.8244
Epoch 08 | Train loss: 0.4579 | Validation loss: 0.5068 | Validation accuracy: 0.8236
Epoch 09 | Train loss: 0.4525 | Validation loss: 0.4807 | Validation accuracy: 0.8329
Epoch 10 | Train loss: 0.4475 | Validation loss: 0.4846 | Validation accuracy: 0.8291
